# ExciView tutorial: MgO

This notebook demonstrates the ExciView workflow for an MgO BSE excitation. It covers reciprocal-space inspection, 1% truncation, Mulliken analysis, and average volumetric densities.

The original BSE eigenvector file is not included in this checkout. The notebook therefore runs the post-processing cells on the precomputed files when they are available and shows the exact commands to use after placing the BSE file in this directory.

In [ ]:
from pathlib import Path
import sys
import numpy as np

# Make the repository package importable when this notebook is opened from examples/MgO.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLE_DIR = REPO_ROOT / 'examples' / 'MgO'
CUBE_DIR = REPO_ROOT / 'examples' / 'MgO_cube'
TRUNCATION_THRESHOLD = 0.01  # Keep contributions with at least 1% weight.
STATE_INDEX = 1
VALENCE_START = 10
CONDUCTION_START = 11
NK = 512
NV = 1
NC = 1
NK_GRID = (8, 8, 8)
print('Repository:', REPO_ROOT)
print('Truncation threshold:', f'{TRUNCATION_THRESHOLD:.0%}')

## 1. Inspect the example files

The Mulliken files are produced by FHI-aims after running the generated input snippet. The cube files in `MgO_cube` are the corresponding precomputed volumetric results.

In [ ]:
print('Mulliken files:', len(list(EXAMPLE_DIR.glob('bandmlk*.out'))))
print('Cube files:', len(list(CUBE_DIR.glob('*.cube'))))
print('Analysis report exists:', (EXAMPLE_DIR / 'exciton_analysis_state_1.dat').exists())
print('BSE input exists:', (EXAMPLE_DIR / 'BSE_eigenvectors.dat').exists())

## 2. Load the BSE state

After copying the ELSI BSE eigenvector file to this directory, set `BSE_FILE` to its name and run this cell. `Exciton` reshapes the selected sparse state into `(k-point, valence band, conduction band)` coefficients.

In [ ]:
from exciview.data import Exciton

BSE_FILE = EXAMPLE_DIR / 'BSE_eigenvectors.dat'
exciton = None
if BSE_FILE.exists():
    exciton = Exciton(STATE_INDEX, NK, NV, NC, NK_GRID)
    exciton.v_start = VALENCE_START
    exciton.c_start = CONDUCTION_START
    exciton.load_from_aims(str(BSE_FILE))
else:
    print(f'Place the BSE file at {BSE_FILE} to run state-dependent analysis.')

## 3. Reciprocal-space analysis

This reports the dominant k point and the valence/conduction band contributions without another FHI-aims calculation.

In [ ]:
from exciview.analysis.reciprocal import analyze_bz_and_bands

if exciton is not None:
    analyze_bz_and_bands(exciton)
else:
    print('Skipped: the BSE state is not present.')

## 4. Generate and analyze Mulliken output

The 1% threshold is applied to normalized k-point weights. Copy `mulliken_snippet.in` into the FHI-aims `control.in`, run FHI-aims, then analyze the resulting `bandmlk*.out` files. The existing MgO files use an offset of 1001.

In [ ]:
from exciview.analysis.mulliken import generate_mulliken_inputs, analyze_mulliken_output

if exciton is not None:
    generate_mulliken_inputs(exciton, threshold=TRUNCATION_THRESHOLD,
                            filename=str(EXAMPLE_DIR / 'mulliken_snippet.in'))
    analyze_mulliken_output(exciton, str(EXAMPLE_DIR / 'bandmlk{}.out'),
                            offset=1001,
                            snippet_file=str(EXAMPLE_DIR / 'mulliken_snippet.in'),
                            v_start=VALENCE_START, c_start=CONDUCTION_START)
else:
    print('Skipped generation because the BSE state is not present.')

## 5. Generate and sum average densities

The volumetric pipeline uses the same 1% cutoff to request only important eigenstate densities. Set `RUN_CUBE_SUM = True` after FHI-aims has produced the cube files to create weighted average hole and electron densities.

In [ ]:
from exciview.analysis.volumetric import generate_cube_inputs, sum_average_density

RUN_CUBE_SUM = False
if exciton is not None:
    cube_snippet = EXAMPLE_DIR / 'cube_snippet.in'
    generate_cube_inputs(exciton, threshold=TRUNCATION_THRESHOLD,
                        filename=str(cube_snippet))
    if RUN_CUBE_SUM:
        # The wildcard absorbs FHI-aims' sequential cube-file prefix.
        cube_pattern = str(CUBE_DIR /
            'cube_*_eigenstate_density_{:05d}_spin_1_k_point_{:04d}.cube')
        sum_average_density(exciton, cube_pattern,
                            control_file=str(CUBE_DIR / 'cube_snippet.in'))
else:
    print('Skipped: load the BSE state before generating cube requests.')

## 6. Inspect the precomputed average cube densities

These files were generated from the significant band/k-point pairs selected at the 1% threshold. ASE is used to read the volumetric grid.

In [ ]:
from exciview.io.cube_tools import safe_read_cube

for kind in ('hole', 'elec'):
    cube_file = CUBE_DIR / f'avg_{kind}_state_1.cube'
    if cube_file.exists():
        data, atoms = safe_read_cube(str(cube_file))
        print(f'{cube_file.name}: shape={data.shape}, integral-like sum={data.sum():.6g}')
    else:
        print(f'Missing {cube_file}')

### FHI-aims handoff

1. Run the generation cell to write `mulliken_snippet.in` or `cube_snippet.in`.
2. Copy the generated commands into `control.in` and run FHI-aims.
3. Return to this notebook and run the matching analysis cell.

The truncation threshold in this tutorial is **1%**, represented in Python as `0.01`.